# GO Reasoning Dataset

### Step 1: Make a fasta file of all proteins in our train + test set

In [1]:
import pandas as pd
import numpy as np
import os
from datasets import load_dataset

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

all = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])['protein_id']
all.drop_duplicates(inplace=True)
all.reset_index(drop=True, inplace=True)

## Fetching Uniprot info

In [35]:
import requests
import pandas as pd
from time import sleep
from tqdm import tqdm
import os

def fetch_uniprot_dates_large(all_accessions, batch_size=500, output_file="uniprot_dates.tsv"):
    """
    Fetch 'date_created' and 'date_modified' for a list of UniProt protein IDs in batches.
    Saves results incrementally after each batch.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    fields = ["accession", "date_created", "date_modified"]

    # Prepare output file with header
    if not os.path.exists(output_file):
        with open(output_file, "w") as f:
            f.write("Accession\tDateCreated\tDateModified\n")

    # Break IDs into batches
    batches = [all_accessions[i:i + batch_size] for i in range(0, len(all_accessions), batch_size)]

    for batch_num, batch in enumerate(tqdm(batches, desc="Processing batches")):
        query = " OR ".join([f"accession:{acc}" for acc in batch])
        params = {
            "query": query,
            "format": "json",
            "fields": ",".join(fields),
            "size": batch_size
        }

        try:
            r = requests.get(base_url, params=params, timeout=30)
            if r.status_code != 200:
                print(f"Failed with status {r.status_code} — retrying after delay")
                sleep(5)
                continue
            r.raise_for_status()
            data = r.json()

            results = []
            for entry in data.get("results", []):
                acc = entry.get("primaryAccession")
                entry_audit = entry.get("entryAudit", {})
                date_created = entry_audit.get("firstPublicDate", None)
                date_modified = entry_audit.get("lastAnnotationUpdateDate", None)
                results.append([acc, date_created, date_modified])

            # Append batch results to TSV
            df = pd.DataFrame(results, columns=["Accession", "DateCreated", "DateModified"])
            df.to_csv(output_file, sep="\t", index=False, mode="a", header=False)

        except Exception as e:
            print(f"Error on batch {batch_num + 1}: {e}")
            sleep(5)

        sleep(0.1)  # rate-limit buffer

    print(f"✅ Done. Saved to {output_file}")

In [ ]:
fetch_uniprot_dates_large(all.to_list(), batch_size=100)

In [3]:
uniprot = pd.read_csv('uniprot_dates.tsv', sep='\t', names=['uniprot_ac', 'date_added', 'last_modified'])

In [4]:
uniprot

,uniprot_ac,date_added,last_modified
0,A0A087X1C5,2015-04-01,2025-06-18
1,A0A0B4J2F0,2015-09-16,2025-06-18
2,A0A0A7EPL0,2015-12-09,2025-06-18
3,A0A0B4J1G0,2017-01-18,2025-06-18
4,A0A0B4J1N3,2017-12-20,2025-06-18
...,...,...,...
201752,Q6IE36,NaN,NaN
201753,Q75L30,NaN,NaN
201754,Q8N268,NaN,NaN
201755,Q8N402,NaN,NaN


In [5]:
# rows where BOTH are NaN
both_nan = uniprot[uniprot['date_added'].isna() & uniprot['last_modified'].isna()]

In [6]:
both_nan

,uniprot_ac,date_added,last_modified
599,A0A0B4KHH7,NaN,NaN
799,A0A0B4KFU2,NaN,NaN
1099,A0A0H3LHU4,NaN,NaN
1196,A0A0H3LA58,NaN,NaN
1197,A0A0H3LAR3,NaN,NaN
...,...,...,...
201752,Q6IE36,NaN,NaN
201753,Q75L30,NaN,NaN
201754,Q8N268,NaN,NaN
201755,Q8N402,NaN,NaN


In [7]:
len(set(both_nan['uniprot_ac'].tolist()))

97

In [9]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import numpy as np

# Filter rows where BOTH are NaN
both_nan = uniprot[uniprot['date_added'].isna() & uniprot['last_modified'].isna()]

# Get list of accessions
missing_ids = both_nan['uniprot_ac'].tolist()

results = []
merged_acc = {}  # {old_acc: new_acc}

for acc in tqdm(missing_ids, desc="Fetching JSON"):
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.json"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            print(f"⚠️ Failed for {acc}: {r.status_code}")
            continue
        data = r.json()

        acc_id = data.get("primaryAccession", acc)
        if acc_id != acc:
            # Store mapping for second pass
            merged_acc[acc] = acc_id
            print(f"ℹ️ {acc} was merged into {acc_id}")
            continue  # Don’t add results yet, fetch in second loop

        # Handle normal / inactive entries
        if data.get("entryType") == "Inactive":
            date_created = np.nan
            date_modified = "DELETED"
        else:
            entry_audit = data.get("entryAudit", {})
            date_created = entry_audit.get("firstPublicDate", np.nan)
            date_modified = entry_audit.get("lastAnnotationUpdateDate", np.nan)

        results.append([acc, date_created, date_modified])

    except Exception as e:
        print(f"❌ Error for {acc}: {e}")
    time.sleep(0.1)


# Second pass: handle merged accessions
for old_acc, new_acc in tqdm(merged_acc.items(), desc="Fetching merged JSON"):
    url = f"https://rest.uniprot.org/uniprotkb/{new_acc}.json"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            print(f"⚠️ Failed for {new_acc}: {r.status_code}")
            continue
        data = r.json()

        acc_id = data.get("primaryAccession", new_acc)
        if acc_id != new_acc:
            print(f"⚠️ Mismatch: {new_acc} resolved to {acc_id}")

        if data.get("entryType") == "Inactive":
            date_created = np.nan
            date_modified = "DELETED"
        else:
            entry_audit = data.get("entryAudit", {})
            date_created = entry_audit.get("firstPublicDate", np.nan)
            date_modified = entry_audit.get("lastAnnotationUpdateDate", np.nan)

        # Always record using the *original accession* (old_acc)
        results.append([old_acc, date_created, date_modified])

    except Exception as e:
        print(f"❌ Error for {old_acc} → {new_acc}: {e}")
    time.sleep(0.1)

# Convert results to DataFrame
df_results = pd.DataFrame(results, columns=["uniprot_ac", "date_added", "last_modified"])

Fetching JSON:   1%|          | 1/97 [00:00<00:51,  1.87it/s]

ℹ️ A0A0B4KHH7 was merged into Q9VC03


Fetching JSON:   2%|▏         | 2/97 [00:01<00:52,  1.81it/s]

ℹ️ A0A0B4KFU2 was merged into Q8I8V0


Fetching JSON:  29%|██▉       | 28/97 [00:15<00:37,  1.82it/s]

...

Fetching JSON:  51%|█████     | 49/97 [00:26<00:26,  1.80it/s]

ℹ️ Q8T0Q5 was merged into Q9VJ90


Fetching JSON:  59%|█████▉    | 57/97 [00:31<00:22,  1.81it/s]

ℹ️ Q5U4D4 was merged into Q07916


Fetching merged JSON: 100%|██████████| 19/19 [00:10<00:00,  1.73it/s]


In [10]:
df_results

,uniprot_ac,date_added,last_modified
0,A0A0H3LHU4,NaN,DELETED
1,A0A0H3LA58,NaN,DELETED
2,A0A0H3LAR3,NaN,DELETED
3,A0A0H3LCC3,NaN,DELETED
4,A0A0H3LDZ7,NaN,DELETED
...,...,...,...
92,F8VVT9,1999-07-15,2025-06-18
93,Q2PDX8,2025-06-18,2025-06-18
94,Q8I8U9,2025-06-18,2025-06-18
95,Q8T0Q5,2025-06-18,2025-06-18


In [13]:
# Compare sets
both_nan_set = set(both_nan['uniprot_ac'])
df_results_set = set(df_results['uniprot_ac'])

only_in_both_nan = both_nan_set - df_results_set
only_in_results = df_results_set - both_nan_set

print("✅ Equal?", both_nan_set == df_results_set)
print(f"Only in both_nan ({len(only_in_both_nan)}): {only_in_both_nan}")
print(f"Only in df_results ({len(only_in_results)}): {only_in_results}")

✅ Equal? True
Only in both_nan (0): set()
Only in df_results (0): set()


In [14]:
df_results[df_results['date_added'].isna() & df_results['last_modified'].isna()]

,uniprot_ac,date_added,last_modified


In [15]:
# df_results contains the fixed entries for missing ones
# uniprot is your main dataframe

for _, row in df_results.iterrows():
    acc = row["uniprot_ac"]
    date_added = row["date_added"]
    date_modified = row["last_modified"]

    # Find all rows with this accession (handles duplicates)
    mask = uniprot["uniprot_ac"] == acc
    if mask.any():
        if pd.isna(uniprot.loc[mask, "date_added"]).any():
            uniprot.loc[mask, "date_added"] = date_added
        if pd.isna(uniprot.loc[mask, "last_modified"]).any():
            uniprot.loc[mask, "last_modified"] = date_modified

# Save updated dataframe
uniprot.to_csv("uniprot_dates_filled.tsv", sep="\t", index=False)

print("✅ Done. Updated missing entries")

✅ Done. Updated missing entries


In [16]:
uniprot

,uniprot_ac,date_added,last_modified
0,A0A087X1C5,2015-04-01,2025-06-18
1,A0A0B4J2F0,2015-09-16,2025-06-18
2,A0A0A7EPL0,2015-12-09,2025-06-18
3,A0A0B4J1G0,2017-01-18,2025-06-18
4,A0A0B4J1N3,2017-12-20,2025-06-18
...,...,...,...
201752,Q6IE36,NaN,DELETED
201753,Q75L30,NaN,DELETED
201754,Q8N268,NaN,DELETED
201755,Q8N402,NaN,DELETED


## Edit the cafa5 data to reflect this update

In [17]:
ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

In [18]:
uniprot = uniprot.rename(columns={'uniprot_ac':'protein_id'})

In [19]:
# Ensure uniprot has the correct columns
uniprot_small = uniprot[["protein_id", "date_added", "last_modified"]]

# Merge into each CAFA dataframe
cafa5_train = cafa5_train.merge(uniprot_small, on="protein_id", how="left")
cafa5_test = cafa5_test.merge(uniprot_small, on="protein_id", how="left")
cafa5_test_super = cafa5_test_super.merge(uniprot_small, on="protein_id", how="left")

In [20]:
cafa5_train

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,structure_path,string_id,interaction_partners,interpro_ids,interpro_location,date_added,last_modified
0,A0A087X1C5,Putative cytochrome P450 2D7 (EC 1.14.14.1),May be responsible for the metabolism of many ...,Homo sapiens (Human),515.0,Membrane {ECO:0000305}; Multi-pass membrane pr...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,"[GO:0008152, GO:0051716, GO:0009987, GO:007146...","[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0003824, GO:0016491, GO:001670...","[GO:0005575, GO:0110165, GO:0005622, GO:004322...",AF-A0A087X1C5-F1-model_v4.cif,None,None,"[IPR036396, IPR017972, IPR001128, IPR002401, I...","{""IPR036396"": [24, 515], ""IPR050182"": [13, 494...",2015-04-01,2025-06-18
1,A0A0B4J2F0,Protein PIGBOS1 (PIGB opposite strand protein 1),Plays a role in regulation of the unfolded pro...,Homo sapiens (Human),54.0,Mitochondrion outer membrane {ECO:0000269|PubM...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,"[GO:0048583, GO:0050789, GO:0010646, GO:002305...","[GO:0008150, GO:0050789, GO:0065007, GO:004858...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0110165, GO:0005622, GO:004322...",AF-A0A0B4J2F0-F1-model_v4.cif,9606.ENSP00000484893,None,[IPR057394],"{""IPR057394"": [1, 53]}",2015-09-16,2025-06-18
2,A0A0A7EPL0,E4 SUMO-protein ligase PIAL1 (EC 2.3.2.-) (Pro...,E4-type SUMO ligase that promotes SUMO chain f...,Arabidopsis thaliana (Mouse-ear cress),847.0,Nucleus {ECO:0000305}.,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...,"[GO:0008152, GO:0036211, GO:0009628, GO:006500...","[GO:0008150, GO:0008152, GO:0065007, GO:004851...","[GO:0003674, GO:0003824, GO:0016740, GO:014009...",None,AF-A0A0A7EPL0-F1-model_v4.cif,3702.A0A0A7EPL0,"[ESD4, SUMO1, ULP1D, ULP1B, SUMO8, SUMO2, MOM1...","[IPR004181, IPR013083]","{""IPR013083"": [255, 363], ""IPR004181"": [268, 3...",2015-12-09,2025-06-18
3,A0A0B4J1G0,Low affinity immunoglobulin gamma Fc region re...,Receptor for the invariable Fc fragment of imm...,Mus musculus (Mouse),249.0,"Cell membrane {ECO:0000269|PubMed:16039578, EC...",MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...,"[GO:0006898, GO:0051234, GO:0002376, GO:004578...","[GO:0008150, GO:0002376, GO:0009987, GO:006500...","[GO:0003674, GO:0060089, GO:0038023, GO:000488...","[GO:0005575, GO:0110165, GO:0071944, GO:001602...",AF-A0A0B4J1G0-F1-model_v4.cif,10090.ENSMUSP00000077873,"[Gm5150, Vav1, Tyrobp, Cd247, Fcgr2b, Lilra6, ...","[IPR036179, IPR007110, IPR013783, IPR003598, I...","{""IPR013783"": [20, 189], ""IPR036179"": [23, 188...",2017-01-18,2025-06-18
4,A0A0B4J1N3,Protein GPR15LG (Protein GPR15 ligand) (Protei...,Highly cationic protein that has multiple func...,Mus musculus (Mouse),78.0,Secreted {ECO:0000250|UniProtKB:Q6UWK7}.,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...,"[GO:0009605, GO:0051716, GO:0040011, GO:000237...","[GO:0008150, GO:0040011, GO:0002376, GO:000998...","[GO:0003674, GO:0098772, GO:0005488, GO:014067...",None,AF-A0A0B4J1N3-F1-model_v4.cif,10090.ENSMUSP00000136426,[Gpr15],[IPR031713],"{""IPR031713"": [1, 77]}",2017-12-20,2025-06-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133487,V9XTM1,Erythrocyte-binding antigen 175,None,Plasmodium sp. chimpanzee clade C3,631.0,None,FLDSRVNDRKNSSSNNGDLNNCREKRRGMKWECKKKNNTSNYVCIP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,AF-V9XTM1-F1-model_v4.cif,None,None,"[IPR008602, IPR042202]","{""IPR042202"": [6, 488], ""IPR008602"": [43, 503]}",2014-03-19,2025-02-05
133488,W5IDC3,Autonomous cohesin,None,Ruminococcus flavefaciens,201.0,None,MGSSSVTADLNNAVINVDEMNEAFKDVPDLEGEGAHITLSNTTAKP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,AF-W5IDC3-F1-model_v4.cif,None,None,None,NaN,2014-03-19,2025-06-18
133489,Q9YWN5,VP9,None,Banna virus (BAV),283.0,None,MLSETELRALKKLSTTTSRVVGDSTLALPSNVKLSKGEVEKIAVTK.

### Testing

In [21]:
import pandas as pd

# Combine all sets into one DataFrame
all_sets = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
all_sets.reset_index(drop=True, inplace=True)

# Total rows
total = len(all_sets)

# Count how many have both dates present
both_dates = all_sets[all_sets["date_added"].notna() & all_sets["last_modified"].notna()]

# Count how many have at least one missing
one_missing = all_sets[(all_sets["date_added"].isna() & all_sets["last_modified"].notna()) |
                       (all_sets["date_added"].notna() & all_sets["last_modified"].isna())]

# Count how many are completely missing
both_missing = all_sets[all_sets["date_added"].isna() & all_sets["last_modified"].isna()]

# Count how many have last_modified == "deleted"
deleted = all_sets[all_sets["last_modified"] == "DELETED"]

print("📊 Coverage Summary:")
print(f"Total proteins: {total}")
print(f"✅ Both dates present: {len(both_dates)}")
print(f"⚠️ One date missing: {len(one_missing)}")
print(f"❌ Both missing: {len(both_missing)}")
print(f"🗑️ Deleted: {len(deleted)}")

📊 Coverage Summary:
Total proteins: 275603
✅ Both dates present: 275525
⚠️ One date missing: 78
❌ Both missing: 0
🗑️ Deleted: 78


In [22]:
uniprot[uniprot['protein_id'] == 'A0A0B4KHH7']

,protein_id,date_added,last_modified
599,A0A0B4KHH7,2025-06-18,2025-06-18


## Time to Upload

In [27]:
from datasets import Dataset
import json

# Create Hugging Face Datasets
train_hf = Dataset.from_pandas(cafa5_train, preserve_index=False)
test_hf = Dataset.from_pandas(cafa5_test, preserve_index=False)
test_super_hf = Dataset.from_pandas(cafa5_test_super, preserve_index=False)

In [28]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf,
    "test_superset": test_super_hf
})

In [29]:
cafa_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'interpro_ids', 'interpro_location', 'date_added', 'last_modified'],
        num_rows: 133492
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'interpro_ids', 'interpro_location', 'date_added', 'last_modified'],
        num_rows: 264
    })
    test_superset: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'interpro_ids', 'interpro_location', 'date_added', 'last_

In [ ]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Edited CAFA data to include date first annotated and last modified date"
)